## Section 1 — Imports and Configuration

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from pathlib import Path
from io import StringIO
from dotenv import load_dotenv
import sqlalchemy
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import mysql.connector

# Load environment variables from .env file.
# All credentials live in .env and are never hardcoded.
load_dotenv()

DB_HOST     = os.getenv('DB_HOST', 'localhost')
DB_PORT     = os.getenv('DB_PORT', '3306')
DB_NAME     = os.getenv('DB_NAME', 'nashville_analytics')
DB_USER     = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
CENSUS_KEY  = os.getenv('CENSUS_API_KEY')

# Paths
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / 'environment.yml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError('Could not locate project root. Is environment.yml present?')
PROCESSED_DIR     = PROJECT_ROOT / 'data' / 'processed'
REPORTS_DIR       = PROJECT_ROOT / 'reports'
GEOCODED_PATH     = PROCESSED_DIR / 'geocoded_addresses.csv'
DEMOGRAPHICS_PATH = PROCESSED_DIR / 'census_tract_demographics.csv'

# Chart style — consistent with previous notebooks
plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print('Imports loaded.')
print(f'Project root: {PROJECT_ROOT}')
print(f'Census API key present: {bool(CENSUS_KEY)}')

## Section 2 — Database Connection Functions

Same two-function pattern used in all previous notebooks. `get_connection()` for direct operations, `get_engine()` for `pd.read_sql`. The password contains special characters — `quote_plus` handles encoding for the SQLAlchemy URL; the direct connector receives it as a plain argument.

In [ ]:
def get_connection():
    """Return a raw mysql.connector connection.
    Use for direct operations that do not go through pandas.
    Always call conn.close() in a finally block after use.
    """
    return mysql.connector.connect(
        host=DB_HOST,
        port=int(DB_PORT),
        database=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )


def get_engine():
    """Return a SQLAlchemy engine for use with pd.read_sql.
    The password is URL-encoded with quote_plus because SQLAlchemy
    parses the connection string and misinterprets special characters
    like $ and @ if they are passed raw.
    Always call engine.dispose() in a finally block after use.
    """
    encoded_password = quote_plus(DB_PASSWORD)
    url = f'mysql+mysqlconnector://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
    return create_engine(url)


# Verify the connection works before doing any real work.
conn = None
try:
    conn = get_connection()
    print('Database connection successful.')
finally:
    if conn:
        conn.close()

## Section 3 — Load Property Records from MySQL

We load only the columns needed for geocoding and analysis. Loading the full table would bring in columns (assessment values, structural details) that add no value to this notebook and slow the read unnecessarily.

Rows with null `property_address` or `property_city` are excluded — the geocoder requires both fields.

In [ ]:
query = """
    SELECT
        parcel_id,
        property_address,
        property_city,
        sale_date,
        sale_price,
        land_use,
        tax_district,
        sold_as_vacant,
        price_outlier_flag,
        YEAR(sale_date) AS sale_year
    FROM nashville_housing_clean
    WHERE property_address IS NOT NULL
      AND property_city IS NOT NULL
"""

engine = None
try:
    engine = get_engine()
    df = pd.read_sql(query, engine)
finally:
    if engine:
        engine.dispose()

print(f'Rows loaded: {len(df):,}')
print(f'Columns: {list(df.columns)}')
df.head(3)

## Section 4 — Address Preparation for Census Geocoder

The Census Geocoder batch endpoint expects a CSV with no header and four columns in this exact order:
```
unique_id, street_address, city, state, zip
```
Zip code is optional — we do not have it, so we leave the column empty. State is always `TN` (Tennessee) because all Nashville housing records are in Tennessee.

We use `parcel_id` as the unique ID. This lets us join geocoded results back to property records without ambiguity, even if two properties share an address (e.g. condos).

In [ ]:
# Build the geocoder input dataframe.
# parcel_id is the join key — it survives the round trip through the geocoder.
geocode_input = pd.DataFrame({
    'unique_id':      df['parcel_id'],
    'street_address': df['property_address'].str.strip(),
    'city':           df['property_city'].str.strip(),
    'state':          'TN',
    'zip':            ''
})

# Drop duplicate addresses. Many parcels share the same physical address
# (condos, subdivided lots). Geocoding identical addresses multiple times
# wastes API calls — we geocode each unique address once and join results
# back to all matching parcels.
unique_addresses = geocode_input.drop_duplicates(
    subset=['street_address', 'city', 'state']
).reset_index(drop=True)

print(f'Total property records:    {len(df):,}')
print(f'Unique addresses to geocode: {len(unique_addresses):,}')
print(f'Deduplication saved:         {len(df) - len(unique_addresses):,} geocoder calls')
unique_addresses.head(3)

## Section 5 — Batch Geocoding via Census Geocoder

The Census Geocoder batch endpoint accepts up to 10,000 addresses per request as a CSV file upload. We split our unique addresses into batches, send each batch, parse the response, and accumulate results.

The response CSV has the following columns (no header is returned by the API):
```
input_id, input_address, match_status, match_type,
matched_address, coordinates, tiger_line_id, tiger_line_side,
state_fips, county_fips, tract, block
```

We need: `input_id` (to join back), `match_status`, `coordinates` (lat/lon), `state_fips`, `county_fips`, and `tract`. These three FIPS fields combine to form the full 11-digit Census tract FIPS code.

**This cell will take 10–25 minutes to run across all batches.** It saves a checkpoint file after each batch so progress is not lost if a batch fails. If the file `geocoded_addresses.csv` already exists in `data/processed/`, it skips geocoding entirely — you only ever run this once.

In [ ]:
GEOCODER_URL  = 'https://geocoding.geo.census.gov/geocoder/geographies/addressbatch'
BATCH_SIZE    = 10_000
GEOCODER_COLS = [
    'input_id', 'input_address', 'match_status', 'match_type',
    'matched_address', 'coordinates', 'tiger_line_id', 'tiger_line_side',
    'state_fips', 'county_fips', 'tract', 'block'
]


def geocode_batch(batch_df: pd.DataFrame, batch_num: int) -> pd.DataFrame:
    """Send one batch of addresses to the Census Geocoder.
    Returns a dataframe of parsed results for this batch.
    The response CSV uses double-quoting throughout — quotechar and
    quoting parameters are set explicitly to handle this correctly.
    """
    csv_buffer = StringIO()
    batch_df.to_csv(csv_buffer, index=False, header=False)
    csv_bytes = csv_buffer.getvalue().encode('utf-8')

    response = requests.post(
        GEOCODER_URL,
        files={'addressFile': ('addresses.csv', csv_bytes, 'text/csv')},
        data={
            'benchmark': 'Public_AR_Current',
            'vintage':   'Census2020_Current',
            'format':    'csv'
        },
        timeout=300
    )

    if response.status_code != 200:
        raise RuntimeError(
            f'Batch {batch_num} failed: HTTP {response.status_code}\n{response.text[:500]}'
        )

    result = pd.read_csv(
        StringIO(response.text),
        header=None,
        names=GEOCODER_COLS,
        dtype=str,
        quotechar='"',          # fields are wrapped in double quotes
        quoting=0,              # 0 = csv.QUOTE_MINIMAL — strip surrounding quotes
        skipinitialspace=True,  # ignore spaces after delimiters
        on_bad_lines='warn'     # warn on malformed rows rather than crashing
    )
    return result

In [ ]:
# --- Main geocoding loop ---

if GEOCODED_PATH.exists():
    print(f'Geocoded file already exists at {GEOCODED_PATH}.')
    print('Skipping geocoding. Delete the file to re-run.')
    geocoded = pd.read_csv(GEOCODED_PATH, dtype=str)

else:
    print('Starting geocoding. This will take 10–25 minutes.')
    print(f'Addresses to process: {len(unique_addresses):,}')
    print(f'Batch size: {BATCH_SIZE:,}  |  Estimated batches: {len(unique_addresses) // BATCH_SIZE + 1}')
    print()

    all_results = []
    n_batches = (len(unique_addresses) // BATCH_SIZE) + 1

    for i in range(n_batches):
        start = i * BATCH_SIZE
        end   = min(start + BATCH_SIZE, len(unique_addresses))
        batch = unique_addresses.iloc[start:end]

        if len(batch) == 0:
            break

        print(f'  Batch {i + 1}/{n_batches}: rows {start:,}–{end:,} ... ', end='', flush=True)
        t0 = time.time()

        try:
            result = geocode_batch(batch, i + 1)
            print(f'    Batch dataframe shape: {result.shape}')
            all_results.append(result)
            elapsed = time.time() - t0
            matched = (result['match_status'].str.upper() == 'MATCH').sum()
            print(f'done in {elapsed:.0f}s , {matched:,}/{len(result):,} matched')
        except Exception as e:
            print(f'FAILED: {e}')
            print('  Saving partial results and stopping.')
            break

        # Brief pause between batches to be polite to the Census server.
        if i < n_batches - 1:
            time.sleep(2)

    geocoded = pd.concat(all_results, ignore_index=True)
    geocoded.to_csv(GEOCODED_PATH, index=False)
    print(f'\nGeocoded results saved to {GEOCODED_PATH}')

print(f'\nTotal records in geocoded file: {len(geocoded):,}')
print(geocoded['match_status'].value_counts())

## Section 6 , Parse Geocoder Results

The geocoder returns coordinates as a single string `lon,lat`. We split that into separate columns. We also construct the full 11-digit Census tract FIPS code by concatenating state FIPS (2 digits), county FIPS (3 digits), and tract (6 digits). This is the standard identifier used to join Census demographic data.

In [ ]:
# Keep only matched records for spatial work.
# Unmatched records are preserved in df (the full property dataset) ,
# they are not dropped from the analysis, just excluded from tract-level findings.
matched = geocoded[geocoded['match_status'].str.upper() == 'MATCH'].copy()

# Split coordinates into lat and lon.
# The geocoder returns 'lon,lat' — note the order: longitude first.
coords = matched['coordinates'].str.split(',', expand=True)
matched['longitude'] = pd.to_numeric(coords[0], errors='coerce')
matched['latitude']  = pd.to_numeric(coords[1], errors='coerce')

# Construct the 11-digit FIPS code: 2-digit state + 3-digit county + 6-digit tract.
# Zero-pad each component to the correct length before concatenating.
matched['state_fips']  = matched['state_fips'].str.zfill(2)
matched['county_fips'] = matched['county_fips'].str.zfill(3)
matched['tract']       = matched['tract'].str.zfill(6)
matched['tract_fips']  = matched['state_fips'] + matched['county_fips'] + matched['tract']

# Match rate summary
total     = len(geocoded)
n_matched = len(matched)
match_rate = n_matched / total * 100

print(f'Total geocoded:  {total:,}')
print(f'Matched:         {n_matched:,}  ({match_rate:.1f}%)')
print(f'Unmatched:       {total - n_matched:,}  ({100 - match_rate:.1f}%)')
print(f'Unique tracts found: {matched["tract_fips"].nunique():,}')
matched[['input_id', 'latitude', 'longitude', 'tract_fips']].head(3)

## Section 7 — Pull ACS Demographic Data

We query the Census ACS 5-Year Estimates API (2016) for Davidson County, TN (FIPS: state=47, county=037). The 5-year estimates are used rather than 1-year because they have smaller margins of error, especially for small tracts with low populations.

Variables requested:
- `B19013_001E` — Median household income (dollars)
- `B17001_002E` — Population below poverty level
- `B17001_001E` — Total population for poverty calculation (denominator)
- `B01003_001E` — Total population
- `B25002_003E` — Vacant housing units
- `B25002_001E` — Total housing units (vacancy rate denominator)

We compute poverty rate and Census-reported vacancy rate ourselves from the component variables rather than pulling a pre-computed rate. This is better practice — pre-computed rates can mask how the denominator was defined.

If `data/processed/census_tract_demographics.csv` already exists, this section skips the API call. You only pull this data once.

In [ ]:
ACS_YEAR     = 2016
STATE_FIPS   = '47'   # Tennessee
COUNTY_FIPS  = '037'  # Davidson County

ACS_VARIABLES = ','.join([
    'NAME',
    'B19013_001E',  # Median household income
    'B17001_002E',  # Population in poverty
    'B17001_001E',  # Total pop (poverty denominator)
    'B01003_001E',  # Total population
    'B25002_003E',  # Vacant housing units
    'B25002_001E',  # Total housing units
])

ACS_URL = (
    f'https://api.census.gov/data/{ACS_YEAR}/acs/acs5'
    f'?get={ACS_VARIABLES}'
    f'&for=tract:*'
    f'&in=state:{STATE_FIPS}%20county:{COUNTY_FIPS}'
    f'&key={CENSUS_KEY}'
)


def pull_acs_demographics() -> pd.DataFrame:
    """Pull ACS tract-level demographics for Davidson County, TN.
    Returns a cleaned dataframe with one row per Census tract.
    """
    response = requests.get(ACS_URL, timeout=60)

    if response.status_code != 200:
        raise RuntimeError(
            f'ACS API request failed: HTTP {response.status_code}\n{response.text[:500]}'
        )

    data = response.json()
    headers = data[0]
    rows    = data[1:]
    raw     = pd.DataFrame(rows, columns=headers)

    # Rename columns to readable names.
    raw = raw.rename(columns={
        'B19013_001E': 'median_hh_income',
        'B17001_002E': 'pop_in_poverty',
        'B17001_001E': 'pop_poverty_denom',
        'B01003_001E': 'total_population',
        'B25002_003E': 'vacant_units',
        'B25002_001E': 'total_units',
    })

    # Convert numeric columns. ACS returns everything as strings.
    # Values of -666666666 indicate missing data in the ACS — replace with NaN.
    numeric_cols = [
        'median_hh_income', 'pop_in_poverty', 'pop_poverty_denom',
        'total_population', 'vacant_units', 'total_units'
    ]
    for col in numeric_cols:
        raw[col] = pd.to_numeric(raw[col], errors='coerce')
        raw[col] = raw[col].replace(-666666666, np.nan)

    # Compute derived rates.
    raw['poverty_rate']  = raw['pop_in_poverty']  / raw['pop_poverty_denom']
    raw['vacancy_rate']  = raw['vacant_units']     / raw['total_units']

    # Build 11-digit tract FIPS to match geocoder output.
    raw['tract_fips'] = (
        raw['state'].str.zfill(2) +
        raw['county'].str.zfill(3) +
        raw['tract'].str.zfill(6)
    )

    return raw[[
        'tract_fips', 'NAME', 'median_hh_income', 'total_population',
        'poverty_rate', 'vacancy_rate', 'vacant_units', 'total_units'
    ]].copy()


if DEMOGRAPHICS_PATH.exists():
    print(f'Demographics file already exists at {DEMOGRAPHICS_PATH}.')
    print('Skipping ACS pull. Delete the file to re-run.')
    demographics = pd.read_csv(DEMOGRAPHICS_PATH)
else:
    print('Pulling ACS demographics from Census API...')
    demographics = pull_acs_demographics()
    demographics.to_csv(DEMOGRAPHICS_PATH, index=False)
    print(f'Saved to {DEMOGRAPHICS_PATH}')

print(f'\nTracts in demographics file: {len(demographics):,}')
print(f'Median income range: ${demographics["median_hh_income"].min():,.0f} – ${demographics["median_hh_income"].max():,.0f}')
demographics.head(3)

## Section 8 — Join Demographics to Property Records

Three datasets come together here:
1. `df` — full property records from MySQL
2. `matched` — geocoded addresses with `tract_fips` attached
3. `demographics` — ACS tract-level variables

The join sequence is: property records → geocoded results (on `parcel_id`) → demographics (on `tract_fips`). Left joins are used throughout so unmatched records are retained in the dataset and their exclusion from tract-level analysis is explicit and documented.

In [ ]:
# Step 1: attach tract_fips to each parcel.
# matched['input_id'] corresponds to parcel_id in the geocoder input.
parcel_tract = matched[['input_id', 'latitude', 'longitude', 'tract_fips']].rename(
    columns={'input_id': 'parcel_id'}
)

df_joined = df.merge(parcel_tract, on='parcel_id', how='left')

# Step 2: attach demographic variables to each parcel via tract_fips.
df_enriched = df_joined.merge(demographics, on='tract_fips', how='left')

# Separate records with and without demographic coverage.
df_with_demo = df_enriched.dropna(subset=['median_hh_income'])

n_total    = len(df_enriched)
n_with     = len(df_with_demo)
n_without  = n_total - n_with

print(f'Total property records:           {n_total:,}')
print(f'With demographic coverage:        {n_with:,}  ({n_with / n_total * 100:.1f}%)')
print(f'Without demographic coverage:     {n_without:,}  ({n_without / n_total * 100:.1f}%)')
print(f'Unique Census tracts in dataset:  {df_with_demo["tract_fips"].nunique():,}')

df_enriched.head(3)

## Section 9 — Income Quintile Assignment

To analyse how property values vary across the income spectrum, we assign each Census tract to an income quintile (Q1 = lowest 20% of tract median incomes, Q5 = highest 20%). Quintiles are computed across tracts — not across individual property records — so each quintile represents a tier of neighbourhoods, not a tier of individual properties.

This is an important distinction worth understanding for interviews: if you quintile individual properties, you rank properties by their own sale price, which conflates what you are trying to measure. Quintiling tracts by tract-level income keeps the demographic and property variables independent.

In [ ]:
# Assign income quintile at the tract level.
# pd.qcut divides tracts into five equal-sized groups by median household income.
tract_income = (
    df_with_demo
    .drop_duplicates(subset='tract_fips')[['tract_fips', 'median_hh_income']]
    .dropna()
)

tract_income['income_quintile'] = pd.qcut(
    tract_income['median_hh_income'],
    q=5,
    labels=['Q1 (Lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (Highest)']
)

# Merge quintile back into the enriched dataset.
df_with_demo = df_with_demo.merge(
    tract_income[['tract_fips', 'income_quintile']],
    on='tract_fips',
    how='left'
)

# Summary: income range per quintile.
quintile_summary = (
    tract_income.groupby('income_quintile', observed=True)['median_hh_income']
    .agg(['min', 'max', 'count'])
    .rename(columns={'min': 'income_min', 'max': 'income_max', 'count': 'tracts'})
)

print('Income quintile boundaries (tract median household income):')
print(quintile_summary.to_string())

## Section 10 — CPI Adjustment

The same CPI adjustment from Notebook 03 is applied here. All price analysis uses inflation-adjusted values so that nominal price differences across years do not obscure real appreciation differences across income quintiles.

BLS series `CUUR0000SA0`, annual averages (`M13`), base year 2013.

In [ ]:
def fetch_cpi() -> dict:
    """Fetch monthly CPI-U values from the BLS public API and compute
    annual averages. The M13 annual average period is not returned by
    the unauthenticated endpoint, so we compute it from the 12 monthly
    values — which is exactly how BLS derives M13 themselves.
    Returns a dict mapping year (int) to annual average CPI (float).
    """
    url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'
    payload = {
        'seriesid':  ['CUUR0000SA0'],
        'startyear': '2013',
        'endyear':   '2016'
    }
    response = requests.post(url, json=payload, timeout=30)
    response.raise_for_status()

    series = response.json()['Results']['series'][0]['data']

    # Collect all monthly values (M01–M12), excluding M13 if present.
    monthly = [
        (int(row['year']), float(row['value']))
        for row in series
        if row['period'].startswith('M') and row['period'] != 'M13'
    ]

    df_cpi = pd.DataFrame(monthly, columns=['year', 'cpi'])

    # Annual average = mean of the 12 monthly readings for that year.
    annual = df_cpi.groupby('year')['cpi'].mean().to_dict()
    return annual


In [ ]:

cpi_map = fetch_cpi()
BASE_CPI = cpi_map[2013]

print(f'CPI values retrieved: {cpi_map}')
print(f'Base year CPI (2013): {BASE_CPI}')

# Adjust sale prices to 2013 dollars.
df_with_demo['cpi_sale_year']   = df_with_demo['sale_year'].map(cpi_map)
df_with_demo['sale_price_real'] = (
    df_with_demo['sale_price'] * (BASE_CPI / df_with_demo['cpi_sale_year'])
)

print(f'\nRows with valid real price: {df_with_demo["sale_price_real"].notna().sum():,}')

## Section 11 — Analysis: Price Appreciation by Income Quintile

We compare median real sale prices across income quintiles and across sale years to answer: did high-income and low-income neighbourhoods appreciate at the same rate between 2013 and 2016, or did the gap widen?

Price outliers (flagged in the clean table) are excluded from this analysis. Including extreme values would distort median calculations for small tracts.

In [ ]:
# Exclude price outliers and non-residential land use categories
# that would skew residential price analysis.
analysis_df = df_with_demo[
    (df_with_demo['price_outlier_flag'] == 'CLEAN') &
    (df_with_demo['sale_price_real'] > 0) &
    (df_with_demo['income_quintile'].notna())
].copy()

print(f'Records used for price analysis: {len(analysis_df):,}')

# Median real price by quintile and year.
price_by_quintile_year = (
    analysis_df
    .groupby(['income_quintile', 'sale_year'], observed=True)['sale_price_real']
    .median()
    .reset_index()
    .rename(columns={'sale_price_real': 'median_real_price'})
)

# Appreciation: percentage change from 2013 to 2016 per quintile.
price_2013 = price_by_quintile_year[price_by_quintile_year['sale_year'] == 2013].set_index('income_quintile')['median_real_price']
price_2016 = price_by_quintile_year[price_by_quintile_year['sale_year'] == 2016].set_index('income_quintile')['median_real_price']
appreciation = ((price_2016 - price_2013) / price_2013 * 100).reset_index()
appreciation.columns = ['income_quintile', 'real_appreciation_pct']

print('\nReal price appreciation by income quintile (2013–2016):')
print(appreciation.to_string(index=False))

## Section 12 — Analysis: Vacancy Concentration by Income Quintile

We test whether properties sold as vacant are concentrated in lower-income tracts. The `sold_as_vacant` field from the housing data tells us whether each transacted property was vacant at the time of sale. This is different from the ACS vacancy rate, which measures housing units with no current occupant.

Both measures are included. The housing data gives us the vacancy signal at the transaction level; the ACS gives us the structural vacancy rate of the neighbourhood. Comparing them reveals whether transaction-level vacancy follows structural neighbourhood patterns.

In [ ]:
# Transaction-level vacancy rate by income quintile.
# sold_as_vacant is 'Y' or 'N'.
vacancy_df = df_with_demo[df_with_demo['income_quintile'].notna()].copy()
vacancy_df['is_vacant_sale'] = (vacancy_df['sold_as_vacant'] == 'Y').astype(int)

vacancy_by_quintile = (
    vacancy_df
    .groupby('income_quintile', observed=True)
    .agg(
        total_sales      = ('parcel_id', 'count'),
        vacant_sales     = ('is_vacant_sale', 'sum'),
        acs_vacancy_rate = ('vacancy_rate', 'median')
    )
    .reset_index()
)

vacancy_by_quintile['transaction_vacancy_rate'] = (
    vacancy_by_quintile['vacant_sales'] / vacancy_by_quintile['total_sales']
)

print('Vacancy analysis by income quintile:')
print(vacancy_by_quintile.to_string(index=False))

In [ ]:
# Check transaction counts and median prices by quintile and year
# to verify the appreciation figure is not driven by sparse data
volume_check = (
    analysis_df
    .groupby(['income_quintile', 'sale_year'], observed=True)
    .agg(
        count=('parcel_id', 'count'),
        median_price=('sale_price_real', 'median')
    )
    .reset_index()
)
print(volume_check.to_string(index=False))

In [ ]:
# Check how many records were removed by the outlier filter per quintile
outlier_check = (
    df_with_demo[df_with_demo['income_quintile'].notna()]
    .groupby(['income_quintile', 'price_outlier_flag'], observed=True)
    .size()
    .reset_index(name='count')
)
print(outlier_check.to_string(index=False))

## Section 13 — Visualisations

Four charts:
- fig13: Median real sale price by income quintile (2016)
- fig14: Real price appreciation 2013–2016 by income quintile
- fig15: Transaction vacancy rate by income quintile
- fig16: ACS neighbourhood vacancy rate by income quintile

In [ ]:
QUINTILE_ORDER  = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4', 'Q5 (Highest)']
QUINTILE_COLORS = ['#c7522a', '#e8a825', '#74a892', '#008585', '#004f63']

# --- fig13: Median real price by income quintile (2016 only) ---
price_2016_df = price_by_quintile_year[
    price_by_quintile_year['sale_year'] == 2016
].copy()
price_2016_df['income_quintile'] = pd.Categorical(
    price_2016_df['income_quintile'], categories=QUINTILE_ORDER, ordered=True
)
price_2016_df = price_2016_df.sort_values('income_quintile')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    price_2016_df['income_quintile'].astype(str),
    price_2016_df['median_real_price'],
    color=QUINTILE_COLORS
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.set_xlabel('Neighbourhood Income Quintile (Census Tract)')
ax.set_ylabel('Median Real Sale Price (2013 USD)')
ax.set_title('Median Real Sale Price by Neighbourhood Income Quintile — 2016', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig13_price_by_income_quintile.png', dpi=150)
plt.show()
print('fig13 saved.')

In [ ]:
# --- fig14: Real price appreciation 2013–2016 by income quintile ---
appr_df = appreciation.copy()
appr_df['income_quintile'] = pd.Categorical(
    appr_df['income_quintile'], categories=QUINTILE_ORDER, ordered=True
)
appr_df = appr_df.sort_values('income_quintile')

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    appr_df['income_quintile'].astype(str),
    appr_df['real_appreciation_pct'],
    color=QUINTILE_COLORS
)
ax.axhline(0, color='#333333', linewidth=0.8, linestyle='--')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_xlabel('Neighbourhood Income Quintile (Census Tract)')
ax.set_ylabel('Real Price Appreciation (%)')
ax.set_title('Real Price Appreciation 2013–2016 by Neighbourhood Income Quintile', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig14_appreciation_by_quintile.png', dpi=150)
plt.show()
print('fig14 saved.')

In [ ]:
# --- fig15: Transaction vacancy rate by income quintile ---
vac_df = vacancy_by_quintile.copy()
vac_df['income_quintile'] = pd.Categorical(
    vac_df['income_quintile'], categories=QUINTILE_ORDER, ordered=True
)
vac_df = vac_df.sort_values('income_quintile')

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    vac_df['income_quintile'].astype(str),
    vac_df['transaction_vacancy_rate'] * 100,
    color=QUINTILE_COLORS
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_xlabel('Neighbourhood Income Quintile (Census Tract)')
ax.set_ylabel('Transaction Vacancy Rate (%)')
ax.set_title('Share of Properties Sold as Vacant by Neighbourhood Income Quintile', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig15_transaction_vacancy_by_quintile.png', dpi=150)
plt.show()
print('fig15 saved.')

In [ ]:
# --- fig16: ACS neighbourhood vacancy rate by income quintile ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    vac_df['income_quintile'].astype(str),
    vac_df['acs_vacancy_rate'] * 100,
    color=QUINTILE_COLORS
)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_xlabel('Neighbourhood Income Quintile (Census Tract)')
ax.set_ylabel('ACS Neighbourhood Vacancy Rate (%)')
ax.set_title('Census-Reported Neighbourhood Vacancy Rate by Income Quintile', fontsize=12)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'fig16_acs_vacancy_by_quintile.png', dpi=150)
plt.show()
print('fig16 saved.')

## Section 14 — Findings

### Data quality
The Census Geocoder matched 97.6% of unique addresses, providing demographic coverage for 73.4% of total property records (43,819 of 59,701). The gap between geocoding match rate and demographic coverage reflects properties whose geocoded tracts fell outside Davidson County's ACS tract boundaries. Unmatched records are retained in the full dataset but excluded from all tract-level findings below.

### Finding 1 — Price level by income quintile
In 2016, median real sale prices across the four lowest income quintiles were compressed within a narrow $24,000 range, from $181k in Q1 to $205k in Q3, before jumping sharply to $370k in Q5, nearly double the Q4 median. This pattern suggests the Nashville market in this period was not a smooth income-price ladder but two distinct tiers: a broadly accessible lower market and a separate high-income segment operating at a different price level entirely.

### Finding 2 — Price appreciation by income quintile
Lower-income neighbourhoods appreciated substantially faster in real terms over 2013–2016. Q1 tracts appreciated 71.9% compared to 21.8% in Q5, a 3.3x difference. The gradient is consistent and smooth across all five quintiles and held up under volume and outlier stress-testing: Q1 transaction counts grew steadily from 850 to 1,365 year-over-year with no single-year distortion. Despite faster percentage appreciation, the absolute dollar gap between Q1 and Q5 medians remained largely intact, widening slightly from $199k in 2013 to $189k in 2016. Faster percentage growth in lower-income tracts did not translate into meaningful affordability convergence.

### Finding 3 — Transaction vacancy by income quintile
Properties sold as vacant were concentrated in lower-income tracts. Q1 recorded a transaction vacancy rate of 9.4%, 2.7 percentage points above Q4's low of 6.7%. Q5 reversed the downward trend slightly at 7.3%, consistent with investor or estate activity in high-value tracts rather than distress-driven vacancy. The Q5 outlier removal rate (8.4% of records) was the highest of any quintile, confirming that the upper end of the market contains a distinct population of high-value transactions not present elsewhere.

### Finding 4 — ACS neighbourhood vacancy by income quintile
Census-reported structural vacancy falls consistently from 10.5% in Q1 to 4.5% in Q5, confirming that lower-income tracts carry higher baseline vacancy independent of transaction activity. Q3 and Q4 share an identical ACS vacancy rate of 7.6%, suggesting a natural plateau in the middle of the income distribution. The two vacancy measures diverge at Q5: structural vacancy continues falling to its lowest point, while transaction vacancy rises slightly. This divergence indicates that high-income tracts are not distressed but attract a different category of vacant-property sale, likely investment or estate transactions, that the ACS neighbourhood rate does not capture.

### Overall interpretation
Taken together, the four findings describe a Nashville market that treated properties in lower-income neighbourhoods differently across every dimension examined. Lower-income tracts had higher vacancy by both measures, lower absolute prices, and, paradoxically, faster real appreciation. The appreciation finding in particular warrants caution in interpretation: rapid percentage gains from a low base do not close the affordability gap if the absolute dollar difference remains intact. A property appreciating from $105k to $181k generates $76k in nominal value; a Q5 property appreciating from $303k to $370k generates $67k less in absolute terms but still leaves a $189k gap between the two markets. The data is consistent with a market undergoing broad appreciation pressure during this period, with lower-income tracts responding more elastically from a lower starting point rather than genuinely converging toward higher-income neighbourhoods.

## Section 15 — Save Enriched Dataset

The enriched dataset (property records with tract FIPS, demographic variables, and income quintile) is saved to `data/processed/`. This file feeds directly into the Excel executive summary.

In [ ]:
output_cols = [
    'parcel_id', 'property_address', 'property_city', 'sale_date',
    'sale_year', 'sale_price', 'sale_price_real', 'land_use',
    'tax_district', 'sold_as_vacant', 'price_outlier_flag',
    'latitude', 'longitude', 'tract_fips',
    'median_hh_income', 'total_population', 'poverty_rate',
    'vacancy_rate', 'income_quintile'
]

# Only include columns that exist after the join.
available_cols = [c for c in output_cols if c in df_with_demo.columns]
df_output = df_with_demo[available_cols].copy()

output_path = PROCESSED_DIR / 'nashville_enriched.csv'
df_output.to_csv(output_path, index=False)

print(f'Enriched dataset saved: {output_path}')
print(f'Rows: {len(df_output):,}  |  Columns: {len(df_output.columns)}')